In [ ]:
import json, time, tqdm, os, pickle, regex as rx, pandas as pd, random

from datetime import datetime
from collections import Counter
from openai import OpenAI
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from islab.nlp.cancer_registry.datasets.utils import CRCPlainTextReader
from dateutil.parser import parse
from pathlib import Path
from sklearn.metrics import classification_report
from llama_index.core.prompts.base import PromptTemplate
from llama_index.core import VectorStoreIndex, Settings, StorageContext, load_index_from_storage
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.callbacks import CallbackManager, LlamaDebugHandler
from llama_index.core.schema import NodeWithScore
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.tools.tool_spec.base import BaseToolSpec
from islab.nlp.cancer_registry.agents.llamaindex import CancerRegistryReActAgent
from llama_index.core.llms import ChatMessage
from llama_index.core.callbacks.schema import EventPayload, CBEventType
from llama_index.core.callbacks.base import BaseCallbackHandler

In [ ]:
# HospitalID = "KMUH"
# HospitalID = "CMUH"
# HospitalID = "KAFGH"
HospitalID = "PPH"

# retriever_models = {"BM25", "GIST-large-Embedding-v0", "multilingual-e5-large-instruct", "b1ade-embed", "bge-fr-en", "Solon-embeddings-large-0.1", "GIST-small-Embedding-v0", "GIST-Embedding-v0", "mxbai-embed-large-v1", "UAE-Large-V1", "e5-base-4k", "bge-large-en-v1.5"}
retrieverModel = None

retrieverModel = "BM25"
# retrieverModel = "GIST-large-Embedding-v0"
# retrieverModel = "multilingual-e5-large-instruct"
# retrieverModel = "b1ade-embed"
# retrieverModel = "bge-fr-en"
# retrieverModel = "Solon-embeddings-large-0.1"
# retrieverModel = "GIST-small-Embedding-v0"
# retrieverModel = "GIST-Embedding-v0"
# retrieverModel = "mxbai-embed-large-v1"
# retrieverModel = "UAE-Large-V1"
# retrieverModel = "e5-base-4k"
# retrieverModel = "bge-large-en-v1.5"

# llmModel = "Mistral-7B-Instruct-v0.3"
# llmModel = "Llama-3.1-8B-Instruct"
llmModel = "DeepSeek-R1-Distill-Llama-8B"
# llmModel = "DeepSeek-R1-Distill-Qwen-7B"

rerankModel = None

# queryMode = "LLM"
queryMode = "RAG"

chunkSize = 1024
# chunkSize = 512
# chunkSize = 256

topkMerage = False
# topkMerage = True

# top_k = 1
top_k = 3
# top_k = 6

filePatientPath = f'datasets/{HospitalID}/testset'
fileAnsPath = f'datasets/{HospitalID}/Report_Answer.tsv'

In [ ]:
if retrieverModel in {"BM25"}:
    Settings.embed_model = None
else:
    Settings.embed_model = HuggingFaceEmbedding(model_name=f"./.models/Retriever/{retrieverModel}")
Settings.llm = OpenAILike(
    model=llmModel,
    api_base="http://localhost:8000/v1",
    api_key="EMPTY",
    temperature=0.01,
    is_chat_model=False,
    is_function_calling_model=False,
    strict=False,
    max_tokens=256
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f"./.models/LLM/Mistral-7B-v0.3")

In [ ]:
def prepare_time_line_data(docs):
	tl_info = {}
	tl_info['events']=[]
	for doc in docs:
		report = {}
		date = parse(doc.metadata['date'])
		report["start_date"] = {"year": date.year, "month": date.month, "day": date.day}
		report["text"] = {
			"text": doc.text.replace('\n', '<br/>')
		}
		tl_info['events'].append(report)	
	return tl_info

def load_dataset(path):
	reader = CRCPlainTextReader()
	patients = reader.load_data(path, HospitalID)
	patient_list = {}
	for pid in patients.keys():	
		patient_list[Path(pid).stem]=[prepare_time_line_data(patients[pid]), patients[pid]]	
	return patient_list

def load_gold_answers(path):
	df = pd.read_csv(path, sep='\t')
	return df

patient_list = load_dataset(filePatientPath)
gold_ans = load_gold_answers(fileAnsPath)

In [ ]:
ignore_list= ["f02af613e39ea56439b0f07fb8ea0df6", "f04b8e681dd034ffb824ee02bd93430b", "f04b8e681dd034ffb824ee02bd93430b",
              "f071e89186538c0585b39de479bb3375", "f07c69ef6cddb16e3547dbffe378be7e", "0f038609b08ea9930f097d67f210e90",
            "f11e6add274bb0668690d5cfff234e9c","f177a11156aaa36944e25fcaf40093d1","f192b5f188924d2cccce6b937e7574ad",
            "f1d68b0e1f2301333e3c45651908c149","f21175384d528a51d59a1ffbeba4d418","f2656a1ca6fb27674b071f086e4bf0df",
            "f2bb03d7f2f892c7f1f734c61950ab66","f309cc0d52849d12a80573ebacdb1710","f314c3a9c3e88761253b092c2157576d",
            "f355449a1620d556250596f416bfc8bd","f37044ce5da8c8148bc85633b9c56d5f", "f3816c6c15c84e837a670848b988fdd4",
            "f3aa8245b1d6fbfd922231317da5f781","f4126d436e7bf23786cc0c2da0b29912","f48c0f7cd40416065a57bd653aed5386",
            "f498820fa6f4d5aab7bec0ffc54c8718","f4a4effc1ad5abb8db7b3dae1fb0bb85","f4cca1760d5730191772b5f4445f88d7",
            "f51013595fdd625b59dcc20afe1ab761","f57d767e6dec9e86601ba6da514c4f4d","f5b102b89e69163aafb619c0661d046b",
            "f5db3c7be6a4e351c8416f08f6fbe7aa","f5e4ce1a4a0de092981846809ac394b0","f67a3b7b00a11b267eb909f672ead14b",
            "f853f6f49c9d40b161bbe21bfb401f7a","f8b85f1445055c343ad02370f8b7a06a","f8be7b390f75f4f43fd214365a94fb4e",
            "f8edffcb3f9d19e7034ead9d544a215c","f971d04e879cc0d4bbb9ee0a2453b5cf","f9d3cec69f0725005087b71e5f45f5fe",
            "fa3c3929227f3eeb748f38629f425dad","fa9ac9dcc3d37f26273aacb963c023c5","faf1204786135affc70142fb9932f4c3",
            "faf4c5b21684af29046808829935ccd7","fb25af4d3985924f61edf7c2b7261891","fd188be319a4c3fef06cc474a5807144",
            "fd7cd9931a7e7bc4729ef1780613bcd9","ff937072bc143d00708db2cd48e4a7e6","ffccb1d17ea1e16595bcd36e354aa520",
            "f02087e255e12e62bbcb3b7994f2939c"]

def select_registry_item(gold_ans, item_name, ignore_list = []):
    patient_list = load_dataset(filePatientPath)
    gold_ans = load_gold_answers(fileAnsPath)    
    ans = gold_ans[['CHART_NO', item_name]]
    ans[item_name] = ans[item_name].astype(str)
    patient_list = {k: v for k, v in patient_list.items() if k not in ignore_list}
    
    for k in patient_list.keys():
        if HospitalID in {"KMUH"}:
            indexs = ans[ans['CHART_NO'] == k].index
        elif HospitalID in {"PPH", "KAFGH", "CMUH"}:
            indexs = ans[ans['CHART_NO'] == int(k.strip())].index
        value = ans.iloc[indexs][item_name]
        patient_list[k].append({item_name: value.values.tolist()})
    ans = ans[~ans['CHART_NO'].isin(ignore_list)]
    return ans, patient_list

def select_registry_items(gold_ans, item_names, ignore_list = []):
    patient_list = load_dataset(filePatientPath)
    gold_ans = load_gold_answers(fileAnsPath)    
    ans = gold_ans[['CHART_NO'] + item_names]
    ans[item_names] = ans[item_names].astype(str).apply(lambda x: x.str.strip())
    patient_list = {k: v for k, v in patient_list.items() if k not in ignore_list}
    
    for k in patient_list.keys():
        if HospitalID in {"KMUH"}:
            indexs = ans[ans['CHART_NO'] == k].index
        elif HospitalID in {"PPH", "KAFGH", "CMUH"}:
            indexs = ans[ans['CHART_NO'] == int(k.strip())].index
        values = ans.iloc[indexs][item_names]
        patient_list[k].append({item_name: values[item_name].values.tolist() for item_name in item_names})  
    ans = ans[~ans['CHART_NO'].isin(ignore_list)]
    return ans, patient_list

In [ ]:
def create_node_parser(callback_manager = None):
    if chunkSize == 1024:
        chunk_sizes = [1024, 512, 256]
    elif chunkSize == 512:
        chunk_sizes = [512, 256, 128]
    elif chunkSize == 256:
        chunk_sizes = [256, 128]
    return HierarchicalNodeParser.from_defaults(chunk_sizes=chunk_sizes, callback_manager= callback_manager)

node_parser = create_node_parser()

In [ ]:
ccr_histology_template ='''
HISTOLOGY CODING RULES (COLORECTAL CANCER):

General principles:
1. Histology coding MUST be based on pathology reports ONLY.
2. If the document is not a pathology report
   OR does not explicitly diagnose colorectal cancer histology,
   return "NA".
3. Use ONLY exact or synonymous histological terms
   explicitly stated in the document.
4. Do NOT infer histology from grade, stage, imaging, or clinical notes.
5. Select ONE and ONLY ONE allowed histology code.

IMPORTANT EXCLUSIONS:

- Histologic GRADE (e.g., grade I/II/III, well/moderate/poorly differentiated)
  MUST NOT be used to determine histology code.

- Architectural descriptions such as "glandular pattern"
  MUST NOT be interpreted as tubular, villous, or tubulovillous subtype
  unless explicitly stated.

- The presence of adenoma (tubular, villous, tubulovillous)
  does NOT determine histology
  unless carcinoma arising in that adenoma is explicitly stated.

- "Mucinous adenocarcinoma" (code 8480) MUST be used
  ONLY IF the pathology report explicitly states
  that extracellular mucin comprises MORE THAN 50% of the tumor.

- Descriptions such as:
  "adenocarcinoma with mucinous component",
  "focal mucinous differentiation",
  or mucin ≤ 50%
  MUST be coded as Adenocarcinoma, NOS (8140).

Specific selection rules:
- If a specific histologic subtype is stated,
  select the MOST SPECIFIC applicable code.
- If multiple histology types are mentioned,
  choose the highest-grade malignant diagnosis.
- If no subtype is stated,
  use the most general applicable code.

Allowed histology codes:
- 8010: Carcinoma in situ, NOS/ Intraepithelial carcinoma, NOS/ Carcinoma, NOS/ Epithelial tumor, malignant/ Heterotopia-associated carcinoma.
- 8013: Large cell neuroendocrine carcinoma/ Combined large cell neuroendocrine carcinoma.
- 8140: Adenocarcinoma in situ, NOS/ Adenocarcinoma, NOS/ Adenocarcinoma, usual type/ Carcinoma of Skene, Cowper and Littre Glands/ Endolymphatic sac tumor/ Parathyroid carcinoma/ Acinar adenocarcinoma of prostate.
- 8210: Adenocarcinoma in situ in adenomatous polyp/ Adenocarcinoma in situ in polyp, NOS/ Carcinoma in situ in polyp, NOS/ Adenocarcinoma in situ in polypoid adenoma/ Adenocarcinoma in situ in tubular adenoma/ Carcinoma in situ in adenomatous polyp/ Adenocarcinoma in adenomatous polyp/ Adenocarcinoma in polyp, NOS/ Carcinoma in polyp, NOS/ Adenocarcinoma in polypoid adenoma/ Adenocarcinoma in tubular adenoma/ Carcinoma in adenomatous polyp.
- 8211: Tubular adenocarcinoma/ Tubular carcinoma.
- 8213: Serrated adenocarcinoma.
- 8261: Adenocarcinoma in situ in villous adenoma/ Adenocarcinoma in villous adenoma.
- 8263: Adenocarcinoma in situ in tubulovillous adenoma/ Adenocarcinoma in tubulovillous adenoma/ Villoglandular carcinoma/ Villoglandular variant of endometrioid adenocarcinoma/ Papillotubular adenocarcinoma/ Tubulopapillary adenocarcinoma.
- 8410: Sebaceous carcinoma/ Sebaceous adenocarcinoma.
- 8440: Cystadenocarcinoma, NOS.
- 8470: Mucinous cystic neoplasm with high grade dysplasia/ Mucinous cystadenocarcinoma, non-invasive/ Mucinous cystic tumor with high grade dysplasia/ Mucinous cystic neoplasm with high grade intraepithelial neoplasia/ Mucinous cystadenocarcinoma, NOS/ Pseudomucinous cystadenocarcinoma, NOS/ Pseudomucinous adenocarcinoma/ Mucinous cystic tumor with an associated invasive carcinoma/ Mucinous cystic neoplasm with an associated invasive carcinoma/ Papillary mucinous cystadenocarcinoma/ Papillary pseudomucinous cystadenocarcinoma.
- 8480: Mucinous adenocarcinoma/ Acinar adenocarcinoma, mucinous variant/ Colloid adenocarcinoma/ Colloid carcinoma/ Gelatinous adenocarcinoma/ Gelatinous carcinoma/ Mucinous carcinoma/ Mucoid adenocarcinoma/ Mucoid carcinoma/ Mucous adenocarcinoma/ Mucous carcinoma/ Pseudomyxoma peritonei with unknown primary site/ Mucinous tubular and spindle cell carcinoma.
- 8490: Signet ring cell carcinoma/ Signet ring cell adenocarcinoma/ Acinar adenocarcinoma, signet ring-like variant/ Mucinous carcinoma, signet ring cell type/ Poorly cohesive carcinoma/ Signet ring cell/histiocytoid carcinoma.
- 9596: Composite Hodgkin and non-Hodgkin lymphoma/ B-cell lymphoma, unclassifiable, with features intermediate between diffuse large B-cell lymphoma and classical Hodgkin lymphoma.
- NA: if histology cannot be determined.

OUTPUT CONSTRAINT:
- The final value MUST be ONE of the allowed histology codes explicitly listed above.
- Any value NOT included in the allowed histology code list(including text descriptions, synonyms, or partial matches) is INVALID.
'''

ccr_grade_template = '''
GRADE CODING RULES (COLORECTAL CANCER):

General principles:
1. Grade coding MUST be based on a PATHOLOGY REPORT ONLY.
   - Ignore clinical notes, imaging reports,
     operative notes, staging information,
     and treatment descriptions.

2. If the document does NOT explicitly state
   histologic grade or degree of differentiation,
   return "9".

3. Use ONLY explicit statements in the document.
   - Do NOT infer, assume, reinterpret, or summarize.

4. Select ONE and ONLY ONE allowed grade code.

IMPORTANT EXCLUSIONS (VERY IMPORTANT):

- Grade MUST NOT be inferred from:
  - histologic subtype (e.g., mucinous, signet ring)
  - tumor architecture (e.g., glandular pattern)
  - tumor stage (pT, pN, pM)
  - lymph node status or metastasis
  - tumor aggressiveness statements

- The presence of adenoma, carcinoma in situ,
  or invasive carcinoma alone
  does NOT determine grade.

Allowed grade codes:
- 1: Well differentiated/ low grade/ Grade I.
- 2: Moderately differentiated/ Grade II.
- 3: Poorly differentiated/ high grade/ Grade III.
- 4: Undifferentiated/ anaplastic/ Grade IV.
- 8: Not applicable.
- 9: Differentiation/grading unclear, no record in medical history.
- X: High grade dysplasia(severe dysplasia)/ Ovary borderline tumor (Low malignant potential) combined with microinvasive carcinoma or/and intraepithelial carcinoma.

OUTPUT CONSTRAINT:
- The final value MUST be ONE of the allowed grade codes explicitly listed above.
- Any value NOT included in the allowed grade code list(including text descriptions, synonyms, or partial matches) is INVALID.
'''

ccr_tumor_size_template = '''
TUMOR SIZE CODING RULES (COLORECTAL CANCER):

General principles:
1. Tumor size MUST be based on a PATHOLOGY REPORT ONLY.
   - Ignore imaging reports, operative notes,
     clinical notes, staging summaries,
     and treatment descriptions.
2. Use ONLY explicit measurements stated
   in the pathology report.
   - Do NOT infer, estimate, assume,
     or reinterpret measurements.
3. Tumor size refers to the MAXIMUM dimension
   of the primary colorectal tumor.
4. Select ONE and ONLY ONE tumor size value.

IMPORTANT EXCLUSIONS:

- Do NOT infer tumor size from:
  - pT stage (e.g., T3, T4)
  - descriptors such as "large", "small",
    or "bulky"
  - margins, invasion depth, or extent
  - lymph node size or metastatic lesions

Specific selection rules:
- If multiple tumor dimensions are explicitly stated
  (e.g., "2.5 × 2.0 × 1.5 cm"),
  select the LARGEST numeric value.
- If tumor size is stated in millimeters (mm),
  convert to centimeters (cm):
    - Divide by 10
    - Keep ONE decimal place only
      (e.g., 18 mm → 1.8 cm)
- If multiple tumor sizes are explicitly stated
  for the SAME primary tumor,
  select the LARGEST value.
- If tumor sizes are mentioned for different lesions
  or No explicit tumor size measurement
  is stated anywhere in the pathology report,
  do NOT infer; return "99.9".

OUTPUT CONSTRAINT:
- The final value MUST be:
  - A numeric value in centimeters (cm), OR "99.9"
  - Do NOT include units in the output value.
  - Do NOT include ranges or multiple values.
'''

ccr_node_examine_template = '''
EXAMINED LYMPH NODES CODING RULES (COLORECTAL CANCER):

General principles:
1. The number of examined lymph nodes MUST be determined
   from a PATHOLOGY REPORT ONLY.
   - Ignore clinical notes, imaging reports,
     operative notes, staging summaries,
     and treatment descriptions.
2. Record ONLY the TOTAL NUMBER of REGIONAL lymph nodes
   EXAMINED by the pathologist,
   regardless of whether the nodes are positive
   or negative for malignancy.
3. Use ONLY explicit numeric information
   stated in the document.
   - Do NOT infer, estimate, assume,
     or derive values from context.
4. Select ONE and ONLY ONE final value.

IMPORTANT EXCLUSIONS:

- Do NOT infer lymph node counts from:
  - TNM stage (pN category)
  - Number of positive lymph nodes alone
  - Statements such as "multiple nodes" or "several nodes"
  - Clinical or radiologic descriptions

- Do NOT count:
  - Distant lymph nodes
    (e.g., para-aortic, supraclavicular)
  - Sentinel lymph nodes
    unless explicitly stated as regional
  - Lymph nodes mentioned without an examined count

COUNTING RULES:

- If the pathology report states a format such as:
  "(0/14)", "(2/16)", (0/12)",
  the TOTAL number examined is the denominator
  (e.g., 14, 16, 12).

- If multiple REGIONAL lymph node groups
  are explicitly reported,
  ADD the examined counts together.

- If regional lymph nodes are examined
  and explicitly reported as "0",
  record the value as "0".

- If the pathology report does NOT explicitly state
  the number of examined REGIONAL lymph nodes,
  return "0".

OUTPUT CONSTRAINT:
- The final output MUST be EXACTLY ONE of the following:
  - a non-negative integer expressed as a string
    (e.g., "0", "9", "14", "18")
  - "0" if the number cannot be determined
- Do NOT output:
  - Ranges (e.g., "10–12")
  - Estimates
  - Text descriptions
  - Multiple values
  - Any explanation outside the required JSON format
'''

ccr_node_positive_template = '''
POSITIVE LYMPH NODES CODING RULES (COLORECTAL CANCER):

General principles:
1. The number of POSITIVE lymph nodes MUST be determined
   from a PATHOLOGY REPORT ONLY.
   - Ignore clinical notes, imaging reports,
     operative notes, staging summaries,
     and treatment descriptions.
2. Record ONLY the TOTAL NUMBER of REGIONAL lymph nodes
   that are POSITIVE for malignancy
   (i.e., involved by metastatic carcinoma).
3. Use ONLY explicit numeric information
   stated in the document.
   - Do NOT infer, estimate, assume,
     or derive values from context.
4. Select ONE and ONLY ONE final value.

IMPORTANT EXCLUSIONS:

- Do NOT infer positive lymph node counts from:
  - TNM stage (pN category)
  - Statements such as "node-positive disease"
  - Percentages without explicit counts
  - Clinical or radiologic descriptions

- Do NOT count:
  - Distant lymph nodes
    (e.g., para-aortic, supraclavicular)
  - Sentinel lymph nodes
    unless explicitly stated as regional
  - Lymph nodes mentioned without
    an explicit POSITIVE count

COUNTING RULES:

- If the pathology report states a format such as:
  "(2/14)", "(3/16)",
  or "positive for malignancy (2/12)",
  the POSITIVE count is the numerator
  (e.g., 2, 3, 2).

- If multiple REGIONAL lymph node groups
  are explicitly reported,
  ADD the POSITIVE counts together.

- Use code "0" ONLY IF the pathology report states a format such as:
  "(0/XX)"

- Use code "98" ONLY IF NO information on lymph nodes in any region is available.

- Use code "99" ONLY IF Regional lymph nodes ARE explicitly mentioned 
  BUT
  the exact number of POSITIVE regional lymph nodes is NOT explicitly stated.

OUTPUT CONSTRAINT (STRICT, NON-NEGOTIABLE):

- The final output MUST be EXACTLY ONE of the following:
  - a non-negative integer expressed as a string
    (e.g., "1", "2", "5")
  - "0"  if the pathology report states a format such as "(0/XX)"
  - "98" If no information on lymph nodes in any region is available
  - "99" if metastatic involvement of regional lymph nodes
         is explicitly stated BUT the positive count
         is NOT recorded

- Do NOT use "99" if:
  - lymph nodes are not mentioned at all
  - lymph nodes are mentioned without any statement
    about metastatic involvement

- Do NOT output:
  - Ranges (e.g., "1–3")
  - Estimates
  - Text descriptions
  - Multiple values
  - Any explanation outside the required JSON format
'''

# ccr_path_tnm_template = '''
# Your task is to extract PATHOLOGICAL TNM staging information
# (PATH_T, PATH_N, PATH_M) for cancer registry abstraction
# STRICTLY based on pathological TNM criteria.

# General principles:
# 1. Use ONLY EXPLICIT PATHOLOGICAL TNM statements.
#    - Accept only pT, pN, pM, or explicitly stated "pathologic T/N/M".
# 2. DO NOT use clinical staging information.
#    - IGNORE cT, cN, cM completely.
# 3. DO NOT infer, assume, reinterpret, calculate, or derive TNM categories.
#    - Tumor size, depth of invasion, lymph node counts,
#      metastasis descriptions, or imaging findings
#      MUST NOT be converted into TNM.
# 4. Each component (PATH_T, PATH_N, PATH_M) is coded independently.
# 5. Select ONE and ONLY ONE code per component.
# 6. When a value is written as "pT3", output ONLY "3".

# PATH_T EXCLUSIONS:
# 1. Use code "8888" (Not applicable) ONLY IF PATH_T is explicitly stated
#    as NOT APPLICABLE or NOT ASSESSED in the pathology report.
# 2. Use code "9999" (Not recorded) ONLY IF PATH_T is completely missing,
#    undocumented, or cannot be found anywhere in the pathology report.

# PATH_N EXCLUSIONS:
# 1. Use code "888" (Not applicable) ONLY IF PATH_N is explicitly stated
#    as NOT APPLICABLE or NOT ASSESSED in the pathology report.
# 2. Use code "999" (Not recorded) ONLY IF PATH_N is completely missing,
#    undocumented, or cannot be found anywhere in the pathology report.

# PATH_M EXCLUSIONS:
# 1. Use code "888" (Not applicable) ONLY IF PATH_M is explicitly stated
#    as NOT APPLICABLE or NOT ASSESSED in the pathology report.
# 2. Use code "999" (Not recorded) ONLY IF PATH_M is completely missing,
#    undocumented, or cannot be found anywhere in the pathology report.
# 3. Use code "B" ONLY IF there is NO mention of distant metastasis
#    anywhere in the pathology report.
#    - In this case, PATH_M is assumed to be "B".
#    - Do NOT infer metastatic status from descriptive findings.

# Pathological TNM staging is determined by pathologists
# and is usually written in the format:
# pT(PATH_T code) pN(PATH_N code) [pM(PATH_M code)].

# UNDER NO CIRCUMSTANCES should you make informed judgments
# or infer TNM staging from descriptive findings.
# '''
ccr_path_tnm_template = '''
Your task is to extract PATHOLOGICAL TNM staging information
(PATH_T, PATH_N, PATH_M) for cancer registry abstraction.

GENERAL PRINCIPLES (HIGHEST PRIORITY):
1. Use ONLY EXPLICIT PATHOLOGICAL TNM statements
   found in the retrieved context.
   - Accept only pT, pN, pM, or explicitly stated
     "pathologic T", "pathologic N", "pathologic M".
2. DO NOT use clinical staging information.
   - IGNORE cT, cN, cM completely.
3. DO NOT infer, assume, reinterpret, calculate,
   or derive TNM categories.
   - Tumor size, depth of invasion,
     lymph node counts, or metastasis descriptions
     MUST NOT be converted into TNM.
4. Each component (PATH_T, PATH_N, PATH_M)
   MUST be extracted independently.
5. Select ONE and ONLY ONE value per component.
6. Ignore spaces, commas, parentheses,
   and extra annotations such as "(2)".

PATH_T RULES:
- Extract the COMPLETE pathological T category
  immediately following "T" or "pT".
- If PATH_T is explicitly stated as
  NOT APPLICABLE or NOT ASSESSED → output "8888".
- If PATH_T is NOT explicitly present
  in the retrieved context → output "9999".
- DO NOT truncate, split, or reinterpret
  alphanumeric T categories.
  (e.g., "pT1a" → output "1A";
         "pTis" → output "IS")

PATH_N RULES:
- Extract the COMPLETE pathological N category
  immediately following "N" or "pN".
- If PATH_N is explicitly stated as
  NOT APPLICABLE or NOT ASSESSED → output "888".
- If PATH_N is NOT explicitly present
  in the retrieved context → output "999".
- DO NOT truncate, split, or reinterpret
  alphanumeric or modified N categories.
  (e.g., "pN1c" → output "1C")

PATH_M RULES:
- Extract the COMPLETE pathological M category
  ONLY if an explicit pathological M statement
  is present, such as:
  • "pM0", "pM1"
  • "pM1a", "pM1b", "pM1c"
  • explicitly stated "pathologic M"
- If PATH_M is no mention of distant metastasis
  → output "B".
- If PATH_M is explicitly stated as
  NOT APPLICABLE or NOT ASSESSED → output "888".
- DO NOT truncate, split, or reinterpret
  alphanumeric or modified N categories.
  (e.g., "pM1a" → output "1A")

Here is the detail definition for the three codes.
- PATH_T: This code indicates the size or extent of the primary tumor.
-- Available codes: X, 0, A, IS, ISU, ISD, ISDC, ISPA, ISLA, 1M, 2, 2A, 2A1, 2A2, 2B, 2C, 2D, 3, 3A, 3B, 1, 1A, 1A1, 1A2, 1B, 1B1, 1B2, 1B3, 1C, 1C1, 1C2, 1C3, 1D, 3C, 3D, 4, 4A, 4B, 4C, 4D, 4E, 8888, 9999
- PATH_N: This code represents the presence and extent of regional lymph node metastasis.
-- Available codes: X, 0, 0A, 0B, 0C, 0D, 1, 1A, 1AS, 1B, 1C, 1M, 2M, 2, 2A, 2B, 2C, 3, 3A, 3B, 3C, 888, 999
- PATH_M: This code indicates the presence of distant metastasis.
-- Available codes: 1, 1A, 1A0, 1A1, 1B, 1B0, 1B1, 1C, 1C0, 1C1, 1D, 1D0, C, CA, CA0, CA1, CB, CB0, CB1, CC, CC0, CC1, CD, CD0, CD1, CE, 888, 999, B.
'''

In [ ]:
# qa_prompt_tmpl_str = """
# Context
# ---------------------
# {context_str}
# ---------------------

# You are a colorectal cancer registry abstraction assistant.

# Task:
# Determine the correct value for the registry field "{FIELD_NAME}" STRICTLY based on the document above.
    
# BASE RULES (MANDATORY):

# 1. Use ONLY explicit information stated in the document.
# 2. Do NOT infer, assume, reinterpret, or summarize.
# 3. If the document does NOT contain sufficient information to determine "{FIELD_NAME}" according to its rules, return "NA".
# 4. Select ONE and ONLY ONE value.
# 5. Ignore information unrelated to "{FIELD_NAME}".

# FIELD-SPECIFIC RULES:
# {FIELD_RULES}

# OUTPUT FORMAT (STRICT JSON, NO EXTRA TEXT):

# {
# "explain": "Reasoning based ONLY on document text",
# "cite": "Exact phrases from the document used",
# "{FIELD_NAME}": "value"
# }
    
# Answer:
# """

qa_prompt_tmpl_str = """
Context
---------------------
{context_str}
---------------------

You are a colorectal cancer registry abstraction assistant.

Task:
Extract PATHOLOGICAL TNM staging values (PATH_T, PATH_N, PATH_M) independently STRICTLY based on the document above.

Each component (PATH_T, PATH_N, PATH_M) MUST be determined independently.
The absence of one component MUST NOT affect the determination of the others.
    
BASE RULES (MANDATORY):

1. Use ONLY explicit information stated in the document.
2. Do NOT infer, assume, reinterpret, or summarize.
3. If the document does NOT contain sufficient information to determine "{FIELD_NAME}" according to its rules.
4. Ignore information unrelated to "{FIELD_NAME}".

FIELD-SPECIFIC RULES:
{FIELD_RULES}

OUTPUT FORMAT (STRICT JSON, NO EXTRA TEXT):

{
"explain": "Reasoning based ONLY on document text",
"cite": "Exact phrases from the document used",
"path_t": "PATH_T code",
"path_n": "PATH_N code",
"path_m": "PATH_M code"
}
    
Answer:
"""
qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

refine_prompt_tmpl_str = """
Original task:
Determine the correct value for the registry field "{FIELD_NAME}".

Original answer:
{existing_answer}

Additional context:
---------------------
{context_msg}
---------------------

FIELD-SPECIFIC RULES:
{FIELD_RULES}

You are refining the answer for ONE registry field ONLY.

This is an EVIDENCE REFINEMENT step, not a re-evaluation.

REFINEMENT RULES (MANDATORY AND STRICT):

1. Field isolation:
   - Focus ONLY on "{FIELD_NAME}".
   - Ignore all information related to other registry fields
     unless explicitly required by the rules of "{FIELD_NAME}".

2. Evidence standard:
   - Use ONLY explicit statements from the document.
   - Do NOT infer, assume, reinterpret, or summarize.

3. Modification constraint:
   - You MUST NOT change the value of "{FIELD_NAME}" IF:
       a) the original answer already contains a VALID value
          from the allowed value set of "{FIELD_NAME}"
       b) AND the new context does NOT explicitly state
          a different and definitive value
          according to the rules of "{FIELD_NAME}".

4. NA protection:
   - If the value of "{FIELD_NAME}" is changed,
     the new value MUST be selected ONLY from
     the allowed code list defined for "FIELD-SPECIFIC RULES".
   - Do NOT output text descriptions, labels,
     synonyms, or any value outside the allowed code set.

5. If the new context does NOT add explicit,
   field-relevant evidence,
   return the ORIGINAL answer unchanged.

OUTPUT REQUIREMENTS:
- Preserve the original JSON schema exactly.
- Do NOT add, remove, or rename fields.
- The value of "{FIELD_NAME}" MUST be a valid code defined in the rules for "FIELD-SPECIFIC RULES".
- Output valid JSON only.

Refined Answer:
"""
refine_prompt_tmpl = PromptTemplate(refine_prompt_tmpl_str)

In [ ]:
# tokens = tokenizer.encode(refine_prompt_tmpl_str)
# len(tokens)

In [ ]:
query_histology = "pathology histology adenocarcinoma"

query_grade = "histologic grade differentiation"

query_tumor_size = "tumor size measurement"

query_examined_node = "lymph nodes were examined"

query_positived_node = "lymph nodes positive for malignancy metastatic"

query_path_tnm = "pT pN pM"

In [ ]:
class RefineTraceHandler(BaseCallbackHandler):
    def __init__(self):
        # ✅ 新版 LlamaIndex 需要的屬性
        self.event_starts_to_ignore = set()
        self.event_ends_to_ignore = set()

        self.steps = []

    # ===== 抽象方法（空實作即可）=====
    def start_trace(self, trace_id: str = None, **kwargs):
        pass

    def end_trace(self, trace_id: str = None, **kwargs):
        pass

    def on_event_start(self, event_type: CBEventType, payload=None, **kwargs):
        pass

    # ===== 我們真正用的 =====
    def on_event_end(self, event_type: CBEventType, payload=None, **kwargs):
        if event_type == CBEventType.LLM:
            self.steps.append({
                "payload": payload,
                "kwargs": kwargs,
            })

import json
import re

def parse_completion_text(text: str):
    """
    Parse a completion that contains:
    - a JSON object at the beginning
    - optional free-text explanation after JSON
    """
    if not text:
        return None, None, None

    text = text.strip()

    # --- Step 1: extract first JSON object ---
    json_match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not json_match:
        return None, None, text  # no JSON found

    json_str = json_match.group(0)

    try:
        json_obj = json.loads(json_str)
    except json.JSONDecodeError:
        json_obj = None

    # --- Step 2: extract explanation text ---
    remaining = text[json_match.end():].strip()

    explanation = None
    if remaining:
        # remove leading "Explanation:" if exists
        explanation = re.sub(
            r"^Explanation\s*:\s*",
            "",
            remaining,
            flags=re.IGNORECASE
        ).strip()

    return json_obj, explanation, text

def parse_pathTNM_completion_text(text):
    """
    Parse a completion that contains:
    - a JSON object at the beginning
    - optional free-text explanation after JSON
    """
    if not text:
        return None, None, None

    text = text.strip()

    # --- Step 1: extract first JSON object ---
    json_match = re.search(r"\{[\s\S]*\}", text)
    if not json_match:
        return None, None, text  # no JSON found

    json_str = json_match.group(0)

    try:
        json_obj = json.loads(json_str)
    except json.JSONDecodeError:
        json_obj = None

    # --- Step 2: extract explanation text ---
    remaining = text[json_match.end():].strip()

    explanation = None
    if remaining:
        # remove leading "Explanation:" if exists
        explanation = re.sub(
            r"^Explanation\s*:\s*",
            "",
            remaining,
            flags=re.IGNORECASE
        ).strip()

    return json_obj, explanation, text

In [ ]:
class CRCRetrievalEngineToolSpec(BaseToolSpec):
    def __init__(
        self,
        doc,
        idx,
        nodes,
        similarity_top_k = top_k,
        rerank = None
    ) -> None:
        """Initialize with parameters."""
        file_path = Path(doc.metadata["file_path"])
        file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)        
        vi_out_path = f".data/{HospitalID}/{chunkSize}/{retrieverModel}/indexes/{file_id}_{idx}"

        self.nodes = nodes

        if not os.path.exists(vi_out_path):
            leaf_nodes = get_leaf_nodes(nodes)        
            storage_context = StorageContext.from_defaults()
            storage_context.docstore.add_documents(nodes)
            self._index = VectorStoreIndex(
                    leaf_nodes,
                    storage_context=storage_context
                )
            self._index.storage_context.persist(persist_dir=vi_out_path)
        else:
            self._index = load_index_from_storage(
                StorageContext.from_defaults(persist_dir=vi_out_path),
            )
        self.similarity_top_k = similarity_top_k
        if rerankModel != None:
            self.rerank = SentenceTransformerRerank(top_n=3, model=f"./.models/Rerank/{rerankModel}")
    
    def get_retrieved_fn(self, query: str) -> str:
        if retrieverModel == "BM25":
            retriever = BM25Retriever.from_defaults(nodes=self.nodes, similarity_top_k=min(self.similarity_top_k, len(self.nodes)))
        else:
            retriever = self._index.as_retriever(similarity_top_k=self.similarity_top_k)
        retrieved_nodes = retriever.retrieve(query)

        return retrieved_nodes
    
    def get_agent_histology_fn(self, top_nodes) -> str:
        qa_partial_prompt_tmpl = qa_prompt_tmpl.partial_format(FIELD_RULES = ccr_histology_template, FIELD_NAME = "histology")
        refine_partial_prompt_tmpl = refine_prompt_tmpl.partial_format(FIELD_RULES = ccr_histology_template, FIELD_NAME = "histology")
        if retrieverModel == "BM25":
            retriever = BM25Retriever.from_defaults(nodes=self.nodes, similarity_top_k=min(self.similarity_top_k, len(self.nodes)))
        if rerankModel == None:
            retriever = self._index.as_retriever(similarity_top_k=self.similarity_top_k)
        elif rerankModel != None:
            retriever = self._index.as_retriever(similarity_top_k=self.similarity_top_k)
            reranker_nodes = self.rerank.postprocess_nodes(nodes=top_nodes, query_str=query_histology)
        
        trace_handler = RefineTraceHandler()
        callback_manager = CallbackManager([trace_handler])

        response_synthesizer = get_response_synthesizer(
            response_mode="refine",
            text_qa_template=qa_partial_prompt_tmpl,
            refine_template=refine_partial_prompt_tmpl,
            callback_manager=callback_manager,
        )
        response = response_synthesizer.synthesize(
            query=query_histology,
            nodes=top_nodes
        )

        runs = []
        for i, step in enumerate(trace_handler.steps):
            completion = step["payload"].get(EventPayload.COMPLETION)
            if not completion:
                continue

            json_obj, explanation, raw_text = parse_completion_text(
                completion.text
            )

            runs.append({
                "step": i,
                "phase": "initial" if i == 0 else "refine",
                "parsed_json": json_obj,
                "value": json_obj.get("histology") if json_obj else None,
                "cite": json_obj.get("cite") if json_obj else None,
                "explanation": explanation,
                "raw_completion": raw_text,
            })


        # ===== 6. final structured return =====
        return {
            "final_response": response,
            "runs": runs,
            "source_nodes": [
                {
                    "node_id": n.node.id_,
                    "date": n.node.metadata.get("date"),
                    "text": n.node.text,
                    "score": n.score,
                }
                for n in response.source_nodes
            ],
        }
    
    def get_agent_grade_fn(self, top_nodes) -> str:
        qa_partial_prompt_tmpl = qa_prompt_tmpl.partial_format(FIELD_RULES = ccr_grade_template, FIELD_NAME = "grade")
        refine_partial_prompt_tmpl = refine_prompt_tmpl.partial_format(FIELD_RULES = ccr_grade_template, FIELD_NAME = "grade")
        trace_handler = RefineTraceHandler()
        callback_manager = CallbackManager([trace_handler])

        response_synthesizer = get_response_synthesizer(
            response_mode="refine",
            text_qa_template=qa_partial_prompt_tmpl,
            refine_template=refine_partial_prompt_tmpl,
            callback_manager=callback_manager,
        )
        response = response_synthesizer.synthesize(
            query=query_grade,
            nodes=top_nodes
        )

        runs = []
        for i, step in enumerate(trace_handler.steps):
            completion = step["payload"].get(EventPayload.COMPLETION)
            if not completion:
                continue

            json_obj, explanation, raw_text = parse_completion_text(
                completion.text
            )

            runs.append({
                "step": i,
                "phase": "initial" if i == 0 else "refine",
                "parsed_json": json_obj,
                "value": json_obj.get("grade") if json_obj else None,
                "cite": json_obj.get("cite") if json_obj else None,
                "explanation": explanation,
                "raw_completion": raw_text,
            })


        # ===== 6. final structured return =====
        return {
            "final_response": response,
            "runs": runs,
            "source_nodes": [
                {
                    "node_id": n.node.id_,
                    "date": n.node.metadata.get("date"),
                    "text": n.node.text,
                    "score": n.score,
                }
                for n in response.source_nodes
            ],
        }
    
    def get_agent_tumor_size_fn(self, top_nodes) -> str:
        qa_partial_prompt_tmpl = qa_prompt_tmpl.partial_format(FIELD_RULES = ccr_tumor_size_template, FIELD_NAME = "tumor_size")
        refine_partial_prompt_tmpl = refine_prompt_tmpl.partial_format(FIELD_RULES = ccr_tumor_size_template, FIELD_NAME = "tumor_size")
        trace_handler = RefineTraceHandler()
        callback_manager = CallbackManager([trace_handler])

        response_synthesizer = get_response_synthesizer(
            response_mode="refine",
            text_qa_template=qa_partial_prompt_tmpl,
            refine_template=refine_partial_prompt_tmpl,
            callback_manager=callback_manager,
        )
        response = response_synthesizer.synthesize(
            query=query_tumor_size,
            nodes=top_nodes
        )

        runs = []
        for i, step in enumerate(trace_handler.steps):
            completion = step["payload"].get(EventPayload.COMPLETION)
            if not completion:
                continue

            json_obj, explanation, raw_text = parse_completion_text(
                completion.text
            )

            runs.append({
                "step": i,
                "phase": "initial" if i == 0 else "refine",
                "parsed_json": json_obj,
                "value": json_obj.get("tumor_size") if json_obj else None,
                "cite": json_obj.get("cite") if json_obj else None,
                "explanation": explanation,
                "raw_completion": raw_text,
            })


        # ===== 6. final structured return =====
        return {
            "final_response": response,
            "runs": runs,
            "source_nodes": [
                {
                    "node_id": n.node.id_,
                    "date": n.node.metadata.get("date"),
                    "text": n.node.text,
                    "score": n.score,
                }
                for n in response.source_nodes
            ],
        }
    
    def get_agent_examined_nodes_fn(self, top_nodes) -> str:
        qa_partial_prompt_tmpl = qa_prompt_tmpl.partial_format(FIELD_RULES = ccr_node_examine_template, FIELD_NAME = "examined_nodes")
        refine_partial_prompt_tmpl = refine_prompt_tmpl.partial_format(FIELD_RULES = ccr_node_examine_template, FIELD_NAME = "examined_nodes")
        trace_handler = RefineTraceHandler()
        callback_manager = CallbackManager([trace_handler])

        response_synthesizer = get_response_synthesizer(
            response_mode="refine",
            text_qa_template=qa_partial_prompt_tmpl,
            refine_template=refine_partial_prompt_tmpl,
            callback_manager=callback_manager,
        )
        response = response_synthesizer.synthesize(
            query=query_examined_node,
            nodes=top_nodes
        )

        runs = []
        for i, step in enumerate(trace_handler.steps):
            completion = step["payload"].get(EventPayload.COMPLETION)
            if not completion:
                continue

            json_obj, explanation, raw_text = parse_completion_text(
                completion.text
            )

            runs.append({
                "step": i,
                "phase": "initial" if i == 0 else "refine",
                "parsed_json": json_obj,
                "value": json_obj.get("examined_nodes") if json_obj else None,
                "cite": json_obj.get("cite") if json_obj else None,
                "explanation": explanation,
                "raw_completion": raw_text,
            })


        # ===== 6. final structured return =====
        return {
            "final_response": response,
            "runs": runs,
            "source_nodes": [
                {
                    "node_id": n.node.id_,
                    "date": n.node.metadata.get("date"),
                    "text": n.node.text,
                    "score": n.score,
                }
                for n in response.source_nodes
            ],
        }
    def get_agent_positive_nodes_fn(self, top_nodes) -> str:
        qa_partial_prompt_tmpl = qa_prompt_tmpl.partial_format(FIELD_RULES = ccr_node_positive_template, FIELD_NAME = "positive_nodes")
        refine_partial_prompt_tmpl = refine_prompt_tmpl.partial_format(FIELD_RULES = ccr_node_positive_template, FIELD_NAME = "positive_nodes")
        trace_handler = RefineTraceHandler()
        callback_manager = CallbackManager([trace_handler])

        response_synthesizer = get_response_synthesizer(
            response_mode="refine",
            text_qa_template=qa_partial_prompt_tmpl,
            refine_template=refine_partial_prompt_tmpl,
            callback_manager=callback_manager,
        )
        response = response_synthesizer.synthesize(
            query=query_examined_node,
            nodes=top_nodes
        )

        runs = []
        for i, step in enumerate(trace_handler.steps):
            completion = step["payload"].get(EventPayload.COMPLETION)
            if not completion:
                continue

            json_obj, explanation, raw_text = parse_completion_text(
                completion.text
            )

            runs.append({
                "step": i,
                "phase": "initial" if i == 0 else "refine",
                "parsed_json": json_obj,
                "value": json_obj.get("positive_nodes") if json_obj else None,
                "cite": json_obj.get("cite") if json_obj else None,
                "explanation": explanation,
                "raw_completion": raw_text,
            })


        # ===== 6. final structured return =====
        return {
            "final_response": response,
            "runs": runs,
            "source_nodes": [
                {
                    "node_id": n.node.id_,
                    "date": n.node.metadata.get("date"),
                    "text": n.node.text,
                    "score": n.score,
                }
                for n in response.source_nodes
            ],
        }
    
    def get_agent_path_tnm_fn(self, top_nodes) -> str:
        qa_partial_prompt_tmpl = qa_prompt_tmpl.partial_format(FIELD_RULES = ccr_path_tnm_template, FIELD_NAME = "path_tnm")
        refine_partial_prompt_tmpl = refine_prompt_tmpl.partial_format(FIELD_RULES = ccr_path_tnm_template, FIELD_NAME = "path_tnm")
        trace_handler = RefineTraceHandler()
        callback_manager = CallbackManager([trace_handler])

        response_synthesizer = get_response_synthesizer(
            response_mode="refine",
            text_qa_template=qa_partial_prompt_tmpl,
            refine_template=refine_partial_prompt_tmpl,
            callback_manager=callback_manager,
        )
        response = response_synthesizer.synthesize(
            query=query_path_tnm,
            nodes=top_nodes
        )

        runs = []
        for i, step in enumerate(trace_handler.steps):
            completion = step["payload"].get(EventPayload.COMPLETION)
            if not completion:
                continue
            print("completionText:" + completion.text)
            json_obj, explanation, raw_text = parse_pathTNM_completion_text(
                completion.text
            )

            path_t = json_obj.get("path_t") if json_obj else None
            path_n = json_obj.get("path_n") if json_obj else None
            path_m = json_obj.get("path_m") if json_obj else None

            runs.append({
                "step": i,
                "phase": "initial" if i == 0 else "refine",
                "parsed_json": json_obj,
                "path_t": path_t,
                "path_n": path_n,
                "path_m": path_m,
                "cite": json_obj.get("cite") if json_obj else None,
                "explanation": explanation,
                "raw_completion": raw_text,
            })


        # ===== 6. final structured return =====
        return {
            "final_response": response,
            "runs": runs,
            "source_nodes": [
                {
                    "node_id": n.node.id_,
                    "date": n.node.metadata.get("date"),
                    "text": n.node.text,
                    "score": n.score,
                }
                for n in response.source_nodes
            ],
        }

In [ ]:
def is_number(x):
    try:
        float(x)
        return True
    except (TypeError, ValueError):
        return False

In [ ]:
def build_llm_single_field_artifact(
    llm_parsed_json: dict,
    field_name: str,
    file_id: str,
    analysis_file_path: str,
    answer: str,
    llm_model: str = None,
    top_k: int = None,
):
    # ---------- 1. final answer ----------
    final_answer = {
        "value": None,
        "answer": None,
        "explain": None,
        "cite": None,
    }

    value_match = re.search(rf'"{re.escape(field_name)}"\s*:\s*"([^"]+)"', llm_parsed_json)
    if value_match:
        if field_name == "tumor_size" and HospitalID in {"CMUH", "PPH"}:
            predAnswer = value_match.group(1)
            if is_number(predAnswer):
                final_answer["value"] = str(int(float(predAnswer) * 10))
        else:
            final_answer["value"] = value_match.group(1)
    explain_match = re.search(r'"explain"\s*:\s*"([^"]+)"', llm_parsed_json)
    if explain_match:
        final_answer["explain"] = explain_match.group(1)
    cite_match = re.search(r'"cite"\s*:\s*"([^"]+)"', llm_parsed_json)
    if cite_match:
        final_answer["cite"] = cite_match.group(1)
    final_answer["answer"] = answer[0]

    # ---------- 2. artifact ----------
    artifact = {
        "meta": {
            "doc_id": file_id,
            "registry_field": field_name,
            "timestamp": datetime.utcnow().isoformat(),
            "llm_model": llm_model,
            "top_k": top_k,
        },
        "final_answer": final_answer,
    }

    # ---------- 3. persist ----------
    json_path = f"{analysis_file_path}.json"
    pkl_path = f"{analysis_file_path}"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(artifact, f, ensure_ascii=False, indent=2)

    with open(pkl_path, "wb") as f:
        pickle.dump(artifact, f)

    return artifact

def build_llm_multi_field_artifact(
    llm_parsed_json: dict,
    field_name: str,
    file_id: str,
    analysis_file_path: str,
    answer: str,
    llm_model: str = None,
    top_k: int = None,
):
    # ---------- 1. final answer ----------
    final_answer = {
        "path_t": None,
        "path_n": None,
        "path_m": None,
        "answer": None,
        "explain": None,
        "cite": None,
    }

    path_t_match = re.search(rf'"{re.escape("path_t")}"\s*:\s*"([^"]+)"', llm_parsed_json)
    if path_t_match:
        final_answer["path_t"] = path_t_match.group(1)
    path_n_match = re.search(rf'"{re.escape("path_n")}"\s*:\s*"([^"]+)"', llm_parsed_json)
    if path_n_match:
        final_answer["path_n"] = path_n_match.group(1)
    path_m_match = re.search(rf'"{re.escape("path_m")}"\s*:\s*"([^"]+)"', llm_parsed_json)
    if path_m_match:
        final_answer["path_m"] = path_m_match.group(1)
    explain_match = re.search(r'"explain"\s*:\s*"([^"]+)"', llm_parsed_json)
    if explain_match:
        final_answer["explain"] = explain_match.group(1)
    cite_match = re.search(r'"cite"\s*:\s*"([^"]+)"', llm_parsed_json)
    if cite_match:
        final_answer["cite"] = cite_match.group(1)
    final_answer["answer"] = answer[0]

    # ---------- 2. artifact ----------
    artifact = {
        "meta": {
            "doc_id": file_id,
            "registry_field": field_name,
            "timestamp": datetime.utcnow().isoformat(),
            "llm_model": llm_model,
            "top_k": top_k,
        },
        "final_answer": final_answer,
    }

    # ---------- 3. persist ----------
    json_path = f"{analysis_file_path}.json"
    pkl_path = f"{analysis_file_path}"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(artifact, f, ensure_ascii=False, indent=2)

    with open(pkl_path, "wb") as f:
        pickle.dump(artifact, f)

    return artifact

def build_single_field_artifact(
    agent_result: dict,
    field_name: str,
    file_id: str,
    analysis_file_path: str,
    answer: str,
    llm_model: str = None,
    retriever_model: str = None,
    top_k: int = None,
):
    """
    Build and persist a single-registry-field artifact.

    Parameters
    ----------
    agent_result : dict
        Output from get_agent_xxx_fn(), must contain:
        - "response": LlamaIndex Response
        - "runs": list of LLM runs
        - "source_nodes": list of retrieved nodes (optional if in response)
    field_name : str
        Registry field name, e.g. "histology", "grade", "pT"
    file_id : str
        Unique document id
    analysis_file_path : str
        Base path (without extension) to save artifacts
    """

    # ---------- 1. retrievals ----------
    retrievals = []
    for i, node in enumerate(agent_result.get("source_nodes", [])):
        retrievals.append({
            "rank": i + 1,
            "node_id": node.get("node_id"),
            "score": node.get("score"),
            "text": node.get("text"),
            "date": node.get("date"),
        })

    # ---------- 2. llm runs ----------
    llm_runs = []
    for run in agent_result.get("runs", []):
        llm_runs.append({
            "step": run.get("step"),
            "phase": run.get("phase", "initial" if run.get("step") == 0 else "refine"),
            "value": run.get("value"),
            "cite": run.get("cite"),
            "parsed_json": run.get("parsed_json") or run.get("json"),
            "explanation": run.get("explanation"),
            "raw_completion": run.get("raw_completion"),
        })

    # ---------- 3. final answer ----------
    final_answer = {
        "value": "NA",
        "answer": None,
        "explain": None,
        "cite": None,
    }

    if llm_runs:
        last = llm_runs[-1]
        if field_name == "tumor_size" and HospitalID in {"CMUH", "PPH"}:
            predAnswer = last.get("value", "NA")
            if is_number(predAnswer):
                final_answer["value"] = str(int(float(predAnswer) * 10))
        else:
            final_answer["value"] = last.get("value", "NA")
        parsed = last.get("parsed_json") or {}
        final_answer["answer"] = answer[0]
        final_answer["explain"] = parsed.get("explain")
        final_answer["cite"] = parsed.get("cite")

    # ---------- 4. artifact ----------
    artifact = {
        "meta": {
            "doc_id": file_id,
            "registry_field": field_name,
            "timestamp": datetime.utcnow().isoformat(),
            "llm_model": llm_model,
            "retriever": retriever_model,
            "top_k": top_k,
        },
        "final_answer": final_answer,
        "retrievals": retrievals,
        "llm_runs": llm_runs,
    }

    # ---------- 5. persist ----------
    json_path = f"{analysis_file_path}.json"
    pkl_path = f"{analysis_file_path}"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(artifact, f, ensure_ascii=False, indent=2)

    with open(pkl_path, "wb") as f:
        pickle.dump(artifact, f)

    return artifact

def build_multi_field_artifact(
    agent_result: dict,
    field_name: str,
    file_id: str,
    analysis_file_path: str,
    answer: dict,
    llm_model: str = None,
    retriever_model: str = None,
    top_k: int = None,
):
    # ---------- 1. retrievals ----------
    retrievals = []
    for i, node in enumerate(agent_result.get("source_nodes", [])):
        retrievals.append({
            "rank": i + 1,
            "node_id": node.get("node_id"),
            "score": node.get("score"),
            "text": node.get("text"),
            "date": node.get("date"),
        })

    # ---------- 2. llm runs ----------
    llm_runs = []
    for run in agent_result.get("runs", []):
        llm_runs.append({
            "step": run.get("step"),
            "phase": run.get("phase", "initial" if run.get("step") == 0 else "refine"),
            "path_t": run.get("path_t"),
            "path_n": run.get("path_n"),
            "path_m": run.get("path_m"),
            "cite": run.get("cite"),
            "parsed_json": run.get("parsed_json") or run.get("json"),
            "explanation": run.get("explanation"),
            "raw_completion": run.get("raw_completion"),
        })

    # ---------- 3. final answer ----------
    final_answer = {
        "path_t": None,
        "path_n": None,
        "path_m": None,
        "answer": None,
        "explain": None,
        "cite": None,
    }

    if llm_runs:
        last = llm_runs[-1]
        final_answer["path_t"] = last.get("path_t", "")
        final_answer["path_n"] = last.get("path_n", "")
        final_answer["path_m"] = last.get("path_m", "")
        parsed = last.get("parsed_json") or {}
        final_answer['answer'] = answer
        final_answer["explain"] = parsed.get("explain")
        final_answer["cite"] = parsed.get("cite")

    # ---------- 4. artifact ----------
    artifact = {
        "meta": {
            "doc_id": file_id,
            "registry_field": field_name,
            "timestamp": datetime.utcnow().isoformat(),
            "llm_model": llm_model,
            "retriever": retriever_model,
            "top_k": top_k,
        },
        "final_answer": final_answer,
        "retrievals": retrievals,
        "llm_runs": llm_runs,
    }

    # ---------- 5. persist ----------
    json_path = f"{analysis_file_path}.json"
    pkl_path = f"{analysis_file_path}"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(artifact, f, ensure_ascii=False, indent=2)

    with open(pkl_path, "wb") as f:
        pickle.dump(artifact, f)

    return artifact


In [ ]:
def build_prompt_per_doc(doc, idx, type, answer, target_outcome_list = None):
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)
    analysis_file_path = f".agent/{HospitalID}/LLM_Rule/{llmModel}/{type}/{file_id}_{idx}"
    if os.path.exists(analysis_file_path):
        return pickle.load(open(analysis_file_path, "rb"))
    Path(analysis_file_path).parent.mkdir(parents=True, exist_ok=True)
    
    if type == 'histology':
        FIELD_RULES = ccr_histology_template
        FIELD_NAME = "histology"
    elif type == 'grade':
        FIELD_RULES = ccr_grade_template
        FIELD_NAME = "grade"
    elif type == "tumor_size":
        FIELD_RULES = ccr_tumor_size_template
        FIELD_NAME = "tumor_size"
    elif type == 'examined_nodes':
        FIELD_RULES = ccr_node_examine_template
        FIELD_NAME = "examined_nodes"
    elif type == 'positive_nodes':
        FIELD_RULES = ccr_node_positive_template
        FIELD_NAME = "positive_nodes"
    elif type == 'path_tnm':
        FIELD_RULES = ccr_path_tnm_template
        FIELD_NAME = "path_tnm"
    else:
        raise Exception('Not support yet!')
    
    base_prompt = qa_prompt_tmpl.partial_format(
        context_str = doc.text,
        FIELD_RULES = FIELD_RULES,
        FIELD_NAME = FIELD_NAME
    )
    result = Settings.llm.complete(base_prompt.format()).text

    if type == 'histology':
        artifact = build_llm_single_field_artifact(
                llm_parsed_json=result,
                field_name="histology",
                file_id=file_id,
                analysis_file_path=analysis_file_path,
                answer=answer,
                llm_model=llmModel,
                top_k=top_k,
            )
    elif type == 'grade':
        artifact = build_llm_single_field_artifact(
                llm_parsed_json=result,
                field_name="grade",
                file_id=file_id,
                analysis_file_path=analysis_file_path,
                answer=answer,
                llm_model=llmModel,
                top_k=top_k,
            )
    elif type == 'tumor_size':
        artifact = build_llm_single_field_artifact(
                llm_parsed_json=result,
                field_name="tumor_size",
                file_id=file_id,
                analysis_file_path=analysis_file_path,
                answer=answer,
                llm_model=llmModel,
                top_k=top_k,
            )
    elif type == 'examined_nodes':
        artifact = build_llm_single_field_artifact(
                llm_parsed_json=result,
                field_name="examined_nodes",
                file_id=file_id,
                analysis_file_path=analysis_file_path,
                answer=answer,
                llm_model=llmModel,
                top_k=top_k,
            )
    elif type == 'positive_nodes':
        artifact = build_llm_single_field_artifact(
                llm_parsed_json=result,
                field_name="positive_nodes",
                file_id=file_id,
                analysis_file_path=analysis_file_path,
                answer=answer,
                llm_model=llmModel,
                top_k=top_k,
            )
    elif type == 'path_tnm':
        artifact = build_llm_multi_field_artifact(
                llm_parsed_json=result,
                field_name="path_tnm",
                file_id=file_id,
                analysis_file_path=analysis_file_path,
                answer=answer,
                llm_model=llmModel,
                top_k=top_k,
            )
        
    return artifact

In [ ]:
def build_vote_per_doc(doc, type, allResults, predAnswers, answer, target_outcome_list = None):
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)
    analysis_file_path = f".agent/{HospitalID}/Vote/LLM/{llmModel}/{type}/{file_id}"
    if os.path.exists(analysis_file_path):
        return pickle.load(open(analysis_file_path, "rb"))
        
    Path(analysis_file_path).parent.mkdir(parents=True, exist_ok=True)

    final_answer = {
        "value": "NA",
        "answer": None,
    }

    counter = Counter(predAnswers)
    max_count = max(counter.values())

    top_candidates = [val for val, count in counter.items() if count == max_count]

    if len(top_candidates) == 1:
        chosen_value = top_candidates[0]
    else:
        chosen_value = random.choice(top_candidates)

    candidate_indices = [i for i, v in enumerate(predAnswers) if v == chosen_value]
    chosen_index = random.choice(candidate_indices)

    results = {}
    final_answer["value"] = allResults[chosen_index]['final_answer']['value']
    if type == 'histology' and HospitalID == "CMUH":
        final_answer['answer'] = [answer[0][:4]]
    else:
        final_answer['answer'] = answer[0]

    results["final_answer"] = final_answer

    with open(analysis_file_path+'.json', 'w', encoding="utf-8") as the_file:
        the_file.write(json.dumps(results, ensure_ascii=False, indent=2))
    pickle.dump(results, open(analysis_file_path, "wb"))

    return results

In [ ]:
def build_agent_per_doc(doc, type, nodes, answer, top_nodes, target_outcome_list = None):
    crc_tool_spec = CRCRetrievalEngineToolSpec(doc, 0, nodes)
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)
    if rerankModel == None:
        if top_k == 1:
            analysis_file_path = f".agent/{HospitalID}/Agent/Top-1/{retrieverModel}/{chunkSize}/{llmModel}/{type}/{file_id}"
        elif top_k == 3:
            analysis_file_path = f".agent/{HospitalID}/Agent/RAG/{retrieverModel}/{chunkSize}/{llmModel}/{type}/{file_id}"
        elif top_k == 6:
            analysis_file_path = f".agent/{HospitalID}/Agent/Top-6/{retrieverModel}/{chunkSize}/{llmModel}/{type}/{file_id}"
    elif rerankModel != None:
        analysis_file_path = f".agent/{HospitalID}/Agent/RAG(Rerank)/{rerankModel}/{retrieverModel}/{chunkSize}/{llmModel}/{type}/{file_id}"
    if os.path.exists(analysis_file_path):
        return pickle.load(open(analysis_file_path, "rb"))
        
    Path(analysis_file_path).parent.mkdir(parents=True, exist_ok=True)

    if type == 'histology':
        result = crc_tool_spec.get_agent_histology_fn(top_nodes)

        artifact = build_single_field_artifact(
            agent_result=result,
            field_name="histology",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            retriever_model=retrieverModel,
            top_k=top_k,
        )
    elif type == 'grade':
        result = crc_tool_spec.get_agent_grade_fn(top_nodes)

        artifact = build_single_field_artifact(
            agent_result=result,
            field_name="grade",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            retriever_model=retrieverModel,
            top_k=top_k,
        )
    elif type == 'tumor_size':
        result = crc_tool_spec.get_agent_tumor_size_fn(top_nodes)

        artifact = build_single_field_artifact(
            agent_result=result,
            field_name="tumor_size",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            retriever_model=retrieverModel,
            top_k=top_k,
        )
    elif type == 'examined_nodes':
        result = crc_tool_spec.get_agent_examined_nodes_fn(top_nodes)

        artifact = build_single_field_artifact(
            agent_result=result,
            field_name="examined_nodes",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            retriever_model=retrieverModel,
            top_k=top_k,
        )
    elif type == 'positive_nodes':
        result = crc_tool_spec.get_agent_positive_nodes_fn(top_nodes)

        artifact = build_single_field_artifact(
            agent_result=result,
            field_name="positive_nodes",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            retriever_model=retrieverModel,
            top_k=top_k,
        )
    elif type == 'path_tnm':
        result = crc_tool_spec.get_agent_path_tnm_fn(top_nodes)

        artifact = build_multi_field_artifact(
            agent_result=result,
            field_name="path_tnm",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            retriever_model=retrieverModel,
            top_k=top_k,
        )
    

    return artifact

In [ ]:
def build_analyze_per_doc(doc, idx, type, nodes, answer, target_outcome_list = None):
    crc_tool_spec = CRCRetrievalEngineToolSpec(doc, idx, nodes)

    if type == 'histology':
        retriever = crc_tool_spec.get_retrieved_fn(query_histology)
    elif type == 'grade':
        retriever = crc_tool_spec.get_retrieved_fn(query_grade)
    elif type == 'tumor_size':
        retriever = crc_tool_spec.get_retrieved_fn(query_tumor_size)
    elif type == 'examined_nodes':
        retriever = crc_tool_spec.get_retrieved_fn(query_examined_node)
    elif type == 'positive_nodes':
        retriever = crc_tool_spec.get_retrieved_fn(query_positived_node)
    elif type == 'path_tnm':
        retriever = crc_tool_spec.get_retrieved_fn(query_path_tnm)
    else:
        raise Exception('Not support yet!')
    return retriever

In [ ]:
def build_agent_merage_per_doc(doc, type, nodes, answer, top_nodes, target_outcome_list = None):
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)
    analysis_file_path = f".agent/{HospitalID}/Agent/RAG(Merage)/{retrieverModel}/{chunkSize}/{llmModel}/{type}/{file_id}"
    if os.path.exists(analysis_file_path):
        return pickle.load(open(analysis_file_path, "rb"))
        
    Path(analysis_file_path).parent.mkdir(parents=True, exist_ok=True)

    result = None
    full_text = ""
    for t in top_nodes:
        full_text += t.text + "\n"
    
    if type == 'histology':
        basePrompt = qa_prompt_tmpl.partial_format(context_str = full_text, FIELD_RULES = ccr_histology_template, FIELD_NAME = "histology")
        result = Settings.llm.complete(basePrompt.format()).text

        artifact = build_llm_single_field_artifact(
            llm_parsed_json=result,
            field_name="histology",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            top_k=top_k,
        )
    elif type == 'grade':
        basePrompt = qa_prompt_tmpl.partial_format(context_str = full_text, FIELD_RULES = ccr_grade_template, FIELD_NAME = "grade")
        result = Settings.llm.complete(basePrompt.format()).text

        artifact = build_llm_single_field_artifact(
            llm_parsed_json=result,
            field_name="grade",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            top_k=top_k,
        )
    elif type == 'tumor_size':
        basePrompt = qa_prompt_tmpl.partial_format(context_str = full_text, FIELD_RULES = ccr_tumor_size_template, FIELD_NAME = "tumor_size")
        result = Settings.llm.complete(basePrompt.format()).text

        artifact = build_llm_single_field_artifact(
            llm_parsed_json=result,
            field_name="tumor_size",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            top_k=top_k,
        )
    elif type == 'examined_nodes':
        basePrompt = qa_prompt_tmpl.partial_format(context_str = full_text, FIELD_RULES = ccr_node_examine_template, FIELD_NAME = "examined_nodes")
        result = Settings.llm.complete(basePrompt.format()).text

        artifact = build_llm_single_field_artifact(
            llm_parsed_json=result,
            field_name="examined_nodes",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            top_k=top_k,
        )
    elif type == 'positive_nodes':
        basePrompt = qa_prompt_tmpl.partial_format(context_str = full_text, FIELD_RULES = ccr_node_positive_template, FIELD_NAME = "positive_nodes")
        result = Settings.llm.complete(basePrompt.format()).text

        artifact = build_llm_single_field_artifact(
            llm_parsed_json=result,
            field_name="positive_nodes",
            file_id=file_id,
            analysis_file_path=analysis_file_path,
            answer=answer,
            llm_model=llmModel,
            top_k=top_k,
        )

    return artifact

In [ ]:
def run_experiment(registry_type, one_match_mode = True):
    y_true = []
    y_pred = []
    target_names = {}
    if 'NA' not in target_names:
        target_names['NA'] = len(target_names)
    tp, fp, fn = 0, 0, 0

    for i, k in enumerate(tqdm.tqdm(patient_list.keys())):
        golds = list(patient_list[k][2].values())[0]
        if HospitalID == 'CMUH' and registry_type == 'histology':
            tmp_golds = [list(patient_list[k][2].values())[0][0][:4]].copy()
        # elif HospitalID == 'CMUH' and registry_type == 'tumor_size':
        #     tmp_golds = [list(patient_list[k][2].values())[0][0][:-1] + "." + list(patient_list[k][2].values())[0][0][-1]]
        else:
            tmp_golds = list(patient_list[k][2].values())[0].copy()
        for ans in tmp_golds:
            if ans not in target_names:
                target_names[ans] = len(target_names)
    
        preds = []
        retrieveds = []
        allResults = []
        predAnswers= []
        agentResult = {}

        for idx, doc in enumerate(patient_list[k][1]):
            nodes = node_parser.get_nodes_from_documents([doc])
            if queryMode == "LLM":
                results = build_prompt_per_doc(doc, idx, registry_type, golds)
                allResults.append(results)
                predAnswers.append(results.get(registry_type, None))
            elif queryMode == "RAG":
                retrieved = build_analyze_per_doc(doc, idx, registry_type, nodes, golds)
                retrieveds.append(retrieved)
        if queryMode == "LLM":
            agentResult = build_vote_per_doc(doc, registry_type, allResults, predAnswers, golds)
        elif queryMode == "RAG":
            flat_nodes = [node for group in retrieveds for node in group]
            sorted_nodes = sorted(flat_nodes, key=lambda n: n.score, reverse=True)
            if topkMerage == False:
                if top_k == 1:
                    top_nodes = sorted_nodes[:1]
                elif top_k == 3:
                    top_nodes = sorted_nodes[:3]
                elif top_k == 6:
                    top_nodes = sorted_nodes[:6]
                elif top_k == 10:
                    top_nodes = sorted_nodes[:10]
                agentResult = build_agent_per_doc(doc, registry_type, nodes, golds, top_nodes)
            elif topkMerage == True:
                top_nodes = sorted_nodes[:3]
                agentResult = build_agent_merage_per_doc(doc, registry_type, nodes, golds, top_nodes)

        preds.append(str(agentResult['final_answer']['value']))
        # break
    
        if one_match_mode:
            found_tp = None
            for pred in preds:
                if pred in tmp_golds:
                    pred_id = target_names[pred]
                    gold_id = target_names[tmp_golds[0]]
                    y_pred.append(gold_id)
                    y_true.append(pred_id)
                    tp+=1
                    found_tp = pred
                    break
            if found_tp is None:
                gold_id = target_names[tmp_golds[0]]
                y_true.append(gold_id)

                if len(preds) == 0:
                    pred = 'NA'
                else:
                    pred = preds[-1]
                
                if pred not in target_names:
                    target_names[pred] = len(target_names)
                pred_id = target_names[pred]
                y_pred.append(pred_id)

                fp+=1
                fn+=1
    label_names = []
    final_labels = set(y_true+y_pred)
    for k in target_names.keys():
        if target_names[k] in final_labels:
            label_names.append(str(k))
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division = 0))
    
    print(tp)
    print(fp)
    print(fn)
    print(tp+fn)
    r=tp/(tp+fn) if (tp+fn) > 0 else 0
    p=tp/(tp+fp) if (tp+fp) > 0 else 0
    f=2*r*p/(p+r) if (p+r) > 0 else 0

    print(f"{p}/{r}/{f}")

    return target_names, y_true, y_pred

In [ ]:
def run_experiments(registry_types_name, target_outcome_list, one_match_mode = True):
    y_trues = {}
    y_preds = {}
    target_names = {}
    
    tps, fps, fns = {}, {}, {}

    for i, k in enumerate(tqdm.tqdm(patient_list.keys())):
        golds = patient_list[k][2]
        tmp_golds = {}
        for gold_k in golds.keys():
            tmp_golds[gold_k.lower()] = golds[gold_k].copy()
        # update names
        for type in tmp_golds.keys():
            if type not in target_names:
                target_names[type]= {}
                target_names[type]['NA'] = len(target_names[type])            
    
            for ans in tmp_golds[type]:
                if ans not in target_names[type]:
                    target_names[type][ans] = len(target_names[type])        
        
        preds = {}
        retrieveds = []
        allResults = []
        predAnswers= []
        agentResult = {}
        
        for idx, doc in enumerate(patient_list[k][1]):
            nodes = node_parser.get_nodes_from_documents([doc])
            if queryMode == "LLM":
                results = build_prompt_per_doc(doc, idx, registry_types_name, golds)
                allResults.append(results)
                predAnswers.append([results.get('path_t', None), results.get('path_n', None), results.get('path_m', None)])
            elif queryMode == "RAG":
                retrieved = build_analyze_per_doc(doc, idx, registry_types_name, nodes, golds)
                retrieveds.append(retrieved)
        if queryMode == "LLM":
            agentResult = build_vote_per_doc(doc, registry_types_name, allResults, predAnswers, golds)
        elif queryMode == "RAG":
            flat_nodes = [node for group in retrieveds for node in group]
            sorted_nodes = sorted(flat_nodes, key=lambda n: n.score, reverse=True)
            if topkMerage == False:
                if top_k == 1:
                    top_nodes = sorted_nodes[:1]
                elif top_k == 3:
                    top_nodes = sorted_nodes[:3]
                elif top_k == 6:
                    top_nodes = sorted_nodes[:6]
                elif top_k == 10:
                    top_nodes = sorted_nodes[:10]
                agentResult = build_agent_per_doc(doc, registry_types_name, nodes, golds, top_nodes)
                
        for target_type in target_outcome_list:
            pred = 'NA'
            if target_type in agentResult['final_answer']:
                pred = str(agentResult['final_answer'][target_type])
                if pred not in target_names[target_type]:
                    target_names[target_type][pred] = len(target_names[target_type])  
            else:
                agentResult['final_answer'][target_type.upper()][0] = pred  
            if target_type not in preds:
                preds[target_type] = []
            preds[target_type].append(pred)
        # break
        # one_match_mode == True, 針對每一個病人，只要有一筆答案對就算對了，其他跳過        
        if one_match_mode:
            for target_type in target_outcome_list:
                if target_type not in y_trues:
                    y_trues[target_type]=[]
                    y_preds[target_type]=[]
                    tps[target_type]=0
                    fps[target_type]=0
                    fns[target_type]=0
                found_tp = None
                for gold in tmp_golds[target_type]:
                    if gold in preds[target_type]:
                        gold_id = target_names[target_type][gold]
                        pred_id = target_names[target_type][gold]
                        y_trues[target_type].append(gold_id)
                        y_preds[target_type].append(pred_id)
                        tps[target_type]+=1
                        found_tp = gold
                        break
                if found_tp is None:
                    ans = tmp_golds[target_type][0]
                    ans_id = target_names[target_type][ans]
                    y_trues[target_type].append(ans_id)

                    if len(preds[target_type]) == 0:
                        preds[target_type].append('NA')
                    
                    fns[target_type]+=1
                    fps[target_type]+=1

                    pred = preds[target_type].pop()
                    pred_id = target_names[target_type][pred]
                    y_preds[target_type].append(pred_id)
        else:
            raise Exception('Not checked implementation.')
        
    for target_type in target_outcome_list:
        label_names = []
        final_labels = set(y_trues[target_type]+y_preds[target_type])
        for k in target_names[target_type].keys():
            if target_names[target_type][k] in final_labels:
                label_names.append(k)

        print(classification_report(y_trues[target_type], y_preds[target_type], 
                                    target_names=label_names, zero_division = 0))
    
        print(f'TP: {tps[target_type]}')
        print(f'FP: {fps[target_type]}')
        print(f'FN: {fns[target_type]}')
        #print(tps[target_type]+fns[target_type])
        r=tps[target_type]/(tps[target_type]+fns[target_type])
        p=tps[target_type]/(tps[target_type]+fps[target_type])
        f=2*r*p/(p+r) if (p+r) > 0 else 0
        print(f"P/R/F: {p}/{r}/{f}")

    return target_names, y_trues, y_preds

In [ ]:
# golds, patient_list = select_registry_item(gold_ans, 'HISTOLOGY', ignore_list)
# golds.drop_duplicates()
# target_names, y_true, y_pred = run_experiment('histology')

In [ ]:
# golds, patient_list =  select_registry_item(gold_ans, 'GRADE', ignore_list)
# golds.drop_duplicates()
# target_names, y_true, y_pred = run_experiment('grade')

In [ ]:
if HospitalID != "KAFGH":
    golds, patient_list =  select_registry_item(gold_ans, 'TUMOR_SIZE', ignore_list)
    golds.drop_duplicates()
    target_names, y_true, y_pred = run_experiment('tumor_size')

In [ ]:
# golds, patient_list =  select_registry_item(gold_ans, 'NODE_EXAMINED', ignore_list)
# golds.drop_duplicates()
# target_names, y_true, y_pred = run_experiment('examined_nodes')

In [ ]:
# golds, patient_list =  select_registry_item(gold_ans, 'NODE_POSITIVE', ignore_list)
# golds.drop_duplicates()
# target_names, y_true, y_pred = run_experiment('positive_nodes') 

In [ ]:
# golds, patient_list =  select_registry_items(gold_ans, ['PATH_T', 'PATH_N', 'PATH_M'], ignore_list)
# golds.drop_duplicates()
# target_names, y_true, y_pred = run_experiments('path_tnm', ['path_t', 'path_n', 'path_m'])

In [ ]:
# golds, patient_list = select_registry_item(gold_ans, 'HISTOLOGY', ignore_list)
# golds.drop_duplicates()

# for retrieverModel in retriever_models:
#     print(retrieverModel)
#     globals()["retrieverModel"] = retrieverModel

    # if retrieverModel in {"BM25"}:
    #     Settings.embed_model = None
    # else:
    #     Settings.embed_model = HuggingFaceEmbedding(model_name=f"./.models/Retriever/{retrieverModel}")

#     target_names, y_true, y_pred = run_experiment('histology')